In [ ]:
import pandas as pd
import json
import os
from datetime import datetime
from firecrawl import FirecrawlApp
from pydantic import BaseModel, Field
from typing import List, Dict, Optional
import time

# Initialize Firecrawl app
app = FirecrawlApp(api_key="fc-e38286d15e9a49d69632e03903d800e0")

# Custom exception for rate limiting
class RateLimitException(Exception):
    """Raised when API rate limit is reached"""
    pass

class PathogenEntry(BaseModel):
    """Single pathogen entry with all data from ONE source"""
    pathogen_name: str = Field(..., description="Name of the pathogen (e.g., S. aureus)")
    source_url: str = Field(..., description="Single source URL where ALL the following data originates")
    mic_value: Optional[str] = Field(None, description="MIC value from this source (null if not available)")
    mbc_value: Optional[str] = Field(None, description="MBC value from this source (null if not available)")
    zoi_value: Optional[str] = Field(None, description="Zone of inhibition from this source (null if not available)")
    assay_type: Optional[str] = Field(None, description="Assay type from this source (null if not available)")
    extraction_solvent: Optional[str] = Field(None, description="Extraction solvent from this source (null if not available)")
    preparation_method: Optional[str] = Field(None, description="Preparation method from this source (null if not available)")
    cytotoxicity_cc50: Optional[str] = Field(None, description="CC50 value in μg/mL from this source (null if not available)")
    assay_method: Optional[str] = Field(None, description="Assay method from this source (null if not available)")
    reference_control: Optional[str] = Field(None, description="Reference control used from this source (null if not available)")

class ExtractSchema(BaseModel):
    # Ethnobotanical / geographic metadata
    scientific_name: str
    local_name: str
    family: str
    genus: str
    growth_form: str
    location_found_nepal: str
    traditional_usage: str
    traditional_usage_source_url: str
    cultural_consensus_icf: float

    # Extraction / preparation
    preparation_solvent: str
    preparation_method: str
    preparation_method_source_url: str
    extraction_solvent: str
    compound_class: str
    compound_class_source_url: str
    parts_with_amr: str
    parts_with_amr_source_url: str

    # Antimicrobial data
    pathogen_data: List[PathogenEntry] = Field(
        ..., description="List of pathogen entries. CRITICAL: Each entry must contain ALL data (MIC, MBC, ZOI, assay type, extraction, preparation, CC50, assay method, reference control) from a SINGLE source URL. Do NOT mix data from different sources for the same pathogen. If a field is not available in that specific source, set it to null. Each pathogen entry represents ONE study/source."
    )

    # Synergy data
    synergy_data: List[Dict] = Field(
        ..., description="List of compound pairs found within this plant that show synergistic antimicrobial effects. Each entry should include compound1_name, compound2_name, checkerboard_FIC (≤0.5 for synergy), interaction_class (synergy/antagonism/additive), source URLs, and efflux_inhibition boolean. Focus on compounds naturally occurring together in the same plant parts."
    )

# Configuration
CSV_FILE = "plants_list.csv"
OUTPUT_FILE = "plant_extraction_results.json"
PROCESSED_FILE = "processed_plants.txt"  # Track processed plant names
BATCH_SIZE = 5  # Number of plants to process before showing checkpoint
DELAY_BETWEEN_REQUESTS = 5  # Increased from 2 to 5 seconds to reduce credit burn rate

# Prioritized domains for optimized credit usage
PRIORITY_DOMAINS = [
    "https://www.researchgate.net/publication/",
    "https://pmc.ncbi.nlm.nih.gov/articles/"
]

def load_processed_plants():
    """Load list of already processed plant names from file"""
    if os.path.exists(PROCESSED_FILE):
        with open(PROCESSED_FILE, 'r', encoding='utf-8') as f:
            # Read and strip each line, filter out empty lines
            return set(line.strip() for line in f if line.strip())
    return set()

def save_processed_plant(plant_name):
    """Append a plant name to the processed plants file"""
    with open(PROCESSED_FILE, 'a', encoding='utf-8') as f:
        f.write(f"{plant_name}\n")
    print(f"✓ Added '{plant_name}' to processed list")

def load_existing_results():
    """Load existing results from JSON file if it exists"""
    if os.path.exists(OUTPUT_FILE):
        try:
            with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except json.JSONDecodeError as e:
            print(f"⚠ Warning: Corrupted JSON file detected: {e}")
            print("Starting fresh with empty results")
            # Rename corrupted file
            corrupted_name = f"plant_extraction_results_corrupted_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            os.rename(OUTPUT_FILE, corrupted_name)
            print(f"Renamed corrupted file to: {corrupted_name}")
    
    return {"extraction_date": datetime.now().isoformat(), "plants": [], "processed_count": 0}

def save_results(results):
    """Save results to JSON file"""
    results["last_updated"] = datetime.now().isoformat()
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved results to {OUTPUT_FILE}")

def extract_plant_data(plant_name):
    """Extract data for a single plant with improved error handling"""
    try:
        print(f"\n{'='*80}")
        print(f"Processing: {plant_name}")
        print(f"{'='*80}")
        
        result = app.agent(
            schema=ExtractSchema,
            prompt=(
                f"Extract detailed pharmacological and antimicrobial data for {plant_name}. "
                f"Ethnobotanical and geographic metadata must be specific to Nepal, including "
                f"local name, family, genus, growth form, altitude range, location, distribution, "
                f"traditional usage, and cultural consensus (ICF). "
                f"\n\n"
                f"SEARCH OPTIMIZATION - PRIORITY SOURCES (SEARCH THESE FIRST):\n"
                f"To optimize credit usage, prioritize searching these domains FIRST:\n"
                f"1. https://www.researchgate.net/publication/ (ResearchGate publications)\n"
                f"2. https://pmc.ncbi.nlm.nih.gov/articles/ (PubMed Central articles)\n"
                f"Only search other sources if necessary data is not found in these priority domains.\n"
                f"\n\n"
                f"CRITICAL SOURCING REQUIREMENTS FOR PATHOGEN DATA:\n"
                f"For antimicrobial pathogen data, source global scientific studies for pathogens like "
                f"S. aureus, P. aeruginosa, Acinetobacter baumannii, and E. coli. "
                f"\n\n"
                f"EACH PATHOGEN ENTRY MUST FOLLOW THESE RULES:\n"
                f"1. ALL data fields (MIC, MBC, ZOI, assay type, extraction solvent, preparation method, "
                f"   cytotoxicity CC50, assay method, and reference control) for ONE pathogen entry "
                f"   MUST come from the SAME SINGLE SOURCE/STUDY.\n"
                f"2. The source_url field must be the actual URL of that specific study/paper.\n"
                f"3. DO NOT combine data from different sources for the same pathogen entry.\n"
                f"4. If a specific field (like MBC) is not available in that source, "
                f"   set it to null - DO NOT fill it with data from another source.\n"
                f"5. If you find the same pathogen tested in multiple studies, create SEPARATE pathogen entries "
                f"   for each source, each with its own complete set of data.\n"
                f"6. Verify that the source URL actually contains the data you're extracting.\n"
                f"7. Prefer sources from the priority domains listed above when available.\n"
                f"\n\n"
                f"For synergy data, identify pairs of compounds that naturally occur within {plant_name} "
                f"and demonstrate synergistic antimicrobial activity when combined. Include compound1_name, "
                f"compound2_name, checkerboard_FIC (≤0.5 indicates synergy), interaction_class "
                f"(synergy/antagonism/additive), source URL, and efflux_inhibition (boolean). "
                f"Focus on intra-plant compound interactions."
            ),
            model="spark-1-mini",
        )
        
        # Check if Firecrawl agent returned a failed status
        if hasattr(result, 'get'):
            result_status = result.get('status')
        else:
            # If result is a Pydantic model, convert to dict first
            result_dict = json.loads(result.model_dump_json()) if hasattr(result, 'model_dump_json') else {}
            result_status = result_dict.get('status')
        
        # Handle failed agent responses
        if result_status == 'failed':
            error_text = result.get('error', 'Unknown error') if hasattr(result, 'get') else result_dict.get('error', 'Unknown error')
            error_msg_lower = str(error_text).lower()
            
            print(f"⚠ Firecrawl agent returned failed status: {error_text}")
            
            # Check if it's a credit exhaustion error
            if 'max credits' in error_msg_lower or 'credit' in error_msg_lower:
                print(f"\n{'🛑'*40}")
                print(f"⚠ CREDIT LIMIT REACHED!")
                print(f"Error: {error_text}")
                print(f"{'🛑'*40}\n")
                raise RateLimitException(f"Credit limit reached while processing {plant_name}: {error_text}")
            
            # Check if it's a workspace error (data may be in error message)
            if 'workspace not found' in error_msg_lower:
                print(f"⚠ Workspace error detected - data collection may have succeeded but formatting failed")
                print(f"Saving error for manual review...")
            
            # Return error result
            return {
                "plant_name": plant_name,
                "extraction_timestamp": datetime.now().isoformat(),
                "status": "error",
                "error": error_text,
                "error_type": "firecrawl_agent_failed"
            }
        
        # Convert Pydantic model to dict with JSON serialization
        plant_data = {
            "plant_name": plant_name,
            "extraction_timestamp": datetime.now().isoformat(),
            "status": "success",
            "data": json.loads(result.model_dump_json())  # Convert to JSON-serializable dict
        }
        
        print(f"✓ Successfully extracted data for {plant_name}")
        return plant_data
        
    except Exception as e:
        error_msg = str(e).lower()
        
        # Check for rate limit and credit exhaustion indicators
        if any(indicator in error_msg for indicator in [
            'rate limit', 'rate_limit', 'too many requests', '429', 
            'quota exceeded', 'limit exceeded', 'max credits', 'credit limit'
        ]):
            print(f"\n{'🛑'*40}")
            print(f"⚠ RATE LIMIT OR CREDIT LIMIT REACHED!")
            print(f"Error: {str(e)}")
            print(f"{'🛑'*40}\n")
            raise RateLimitException(f"Rate/credit limit reached while processing {plant_name}: {str(e)}")
        
        # Regular error handling for other exceptions
        print(f"✗ Error processing {plant_name}: {str(e)}")
        return {
            "plant_name": plant_name,
            "extraction_timestamp": datetime.now().isoformat(),
            "status": "error",
            "error": str(e)
        }

def process_plants_in_batches():
    """Main function to process all plants from CSV"""
    # Load plant list from CSV
    df = pd.read_csv(CSV_FILE)
    plant_column = df.columns[0]  # First column contains plant names
    # Strip whitespace from plant names
    plant_list = [str(name).strip() for name in df[plant_column].dropna().tolist()]
    
    print(f"Loaded {len(plant_list)} plants from {CSV_FILE}")
    
    # Load list of already processed plants
    processed_plants = load_processed_plants()
    print(f"Already processed: {len(processed_plants)} plants")
    print(f"Remaining: {len(plant_list) - len(processed_plants)} plants")
    
    # Load existing results
    results = load_existing_results()
    
    # Process plants
    batch_count = 0
    new_plants_in_session = 0
    rate_limit_reached = False
    error_encountered = False
    
    try:
        for i, plant_name in enumerate(plant_list, 1):
            # Skip if already processed
            if plant_name in processed_plants:
                print(f"[{i}/{len(plant_list)}] Skipping {plant_name} (already processed)")
                continue
            
            print(f"\n[{i}/{len(plant_list)}] Processing plant: {plant_name}")
            
            # Extract data
            try:
                plant_data = extract_plant_data(plant_name)
            except RateLimitException as e:
                print(f"\n⚠ Rate/credit limit detected. Stopping further processing.")
                print(f"Current progress has been saved.")
                rate_limit_reached = True
                break
            
            # Check if extraction returned an error status
            if plant_data.get("status") == "error":
                error_type = plant_data.get("error_type", "unknown")
                
                print(f"\n{'⚠'*40}")
                print(f"⚠ ERROR STATUS DETECTED for {plant_name}")
                print(f"Error Type: {error_type}")
                print(f"Error: {plant_data.get('error', 'Unknown error')}")
                
                # For workspace errors, continue processing (data might be recoverable)
                if error_type == "firecrawl_agent_failed":
                    error_msg = str(plant_data.get('error', '')).lower()
                    if 'workspace not found' in error_msg:
                        print(f"⚠ Workspace error - continuing with next plant...")
                        print(f"Note: Error message may contain data for manual recovery")
                        print(f"{'⚠'*40}\n")
                        
                        # Save the error result and continue
                        results["plants"].append(plant_data)
                        results["processed_count"] = len(results["plants"])
                        save_results(results)
                        save_processed_plant(plant_name)
                        processed_plants.add(plant_name)
                        
                        batch_count += 1
                        new_plants_in_session += 1
                        
                        # Add delay before next request
                        if i < len(plant_list):
                            print(f"Waiting {DELAY_BETWEEN_REQUESTS} seconds before next request...")
                            time.sleep(DELAY_BETWEEN_REQUESTS)
                        
                        continue
                
                # For other errors, stop processing
                print(f"Stopping further processing.")
                print(f"{'⚠'*40}\n")
                
                # Still save the error result before stopping
                results["plants"].append(plant_data)
                results["processed_count"] = len(results["plants"])
                save_results(results)
                save_processed_plant(plant_name)
                processed_plants.add(plant_name)
                
                error_encountered = True
                break
            
            # Add to results
            results["plants"].append(plant_data)
            results["processed_count"] = len(results["plants"])
            
            # Save results to JSON
            save_results(results)
            
            # Mark plant as processed (append to processed file)
            save_processed_plant(plant_name)
            processed_plants.add(plant_name)  # Update in-memory set
            
            batch_count += 1
            new_plants_in_session += 1
            
            # Add delay between requests to avoid rate limiting
            if i < len(plant_list):
                print(f"Waiting {DELAY_BETWEEN_REQUESTS} seconds before next request...")
                time.sleep(DELAY_BETWEEN_REQUESTS)
            
            # Optional: Save batch progress
            if batch_count % BATCH_SIZE == 0:
                print(f"\n{'#'*80}")
                print(f"BATCH CHECKPOINT: Processed {new_plants_in_session} new plants in this session")
                print(f"Total plants ever processed: {len(processed_plants)}")
                print(f"Total plants in database: {results['processed_count']}")
                print(f"{'#'*80}\n")
    
    except KeyboardInterrupt:
        print(f"\n\n⚠ Processing interrupted by user (Ctrl+C)")
        print(f"Progress has been saved. You can resume later.")
    
    print(f"\n{'='*80}")
    if rate_limit_reached:
        print(f"PROCESSING STOPPED - RATE/CREDIT LIMIT REACHED")
        print(f"Please wait for credits to reset before resuming processing.")
    elif error_encountered:
        print(f"PROCESSING STOPPED - CRITICAL ERROR ENCOUNTERED")
        print(f"Check the error details above and resolve before resuming.")
    else:
        print(f"EXTRACTION COMPLETE!")
    print(f"New plants processed in this session: {new_plants_in_session}")
    print(f"Total plants ever processed: {len(processed_plants)}")
    print(f"Results saved to: {OUTPUT_FILE}")
    print(f"Processed list saved to: {PROCESSED_FILE}")
    print(f"{'='*80}")
    
    return results

# Start processing
print("Starting batch plant extraction...")
print(f"Configuration:")
print(f"  - CSV File: {CSV_FILE}")
print(f"  - Output File: {OUTPUT_FILE}")
print(f"  - Processed Plants File: {PROCESSED_FILE}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Delay Between Requests: {DELAY_BETWEEN_REQUESTS}s")
print(f"  - Priority Domains: {', '.join(PRIORITY_DOMAINS)}")
print()

results = process_plants_in_batches()


Starting batch plant extraction...
Configuration:
  - CSV File: plants_list.csv
  - Output File: plant_extraction_results.json
  - Processed Plants File: processed_plants.txt
  - Batch Size: 5
  - Delay Between Requests: 5s
  - Priority Domains: https://www.researchgate.net/publication/, https://pmc.ncbi.nlm.nih.gov/articles/

Loaded 499 plants from plants_list.csv
Already processed: 595 plants
Remaining: -96 plants
[1/499] Skipping Aconitum naviculare (already processed)
[2/499] Skipping Aconitum orochryseum (already processed)
[3/499] Skipping Acronema nervosum (already processed)
[4/499] Skipping Actinidia strigosa (already processed)
[5/499] Skipping Actinodaphne obovata (already processed)
[6/499] Skipping Adiantum caudatum (already processed)
[7/499] Skipping Adiantum incisum (already processed)
[8/499] Skipping Adiantum philippense (already processed)
[9/499] Skipping Aeginetia indica (already processed)
[10/499] Skipping Aerides multiflora (already processed)
[11/499] Skipping 